# ACT PETEMUAN 10

Adrian Dermawan Budyanto - 240401020213

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

In [2]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv",
    "telco_churn.csv"
)
print("Dataset berhasil didownload: telco_churn.csv")

df = pd.read_csv('/content/telco_churn.csv')

print("Ukuran dataset:", df.shape)
df.head()
print(df.shape)

Dataset berhasil didownload: telco_churn.csv
Ukuran dataset: (7043, 21)
(7043, 21)


In [3]:
print("Ukuran Dataset :", df.shape)
print()

print(df.info())
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True)*100)

Ukuran Dataset : (7043, 21)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBill

In [4]:
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
    df['TotalCharges'] = df['TotalCharges'].astype(float)
    df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Menghapus kolom ID jika ada karena tidak relevan untuk prediksi
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

# Fitur target diubah menjadi numerik (1 untuk Yes, 0 untuk No)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Pemisahan fitur (X) dan target (y)
X = df.drop(columns=['Churn'])
y = df['Churn']

# Encoding fitur kategorikal menggunakan One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)

# Split data menjadi data latih dan data uji secara stratified
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"\nJumlah data latih: {X_tr.shape[0]}")
print(f"Jumlah data uji: {X_te.shape[0]}")


Jumlah data latih: 5634
Jumlah data uji: 1409


In [5]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)

# Melatih model menggunakan data latih
rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [6]:
pred = rf.predict(X_te)
proba = rf.predict_proba(X_te)[:, 1]

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_te, pred))

roc_auc = roc_auc_score(y_te, proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")


=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Score: 0.8246


In [7]:
# Menampilkan contoh 10 pelanggan pertama dengan probabilitas churn-nya
df_hasil = pd.DataFrame({
    'Aktual': y_te,
    'Prediksi': pred,
    'Probabilitas_Churn': proba
}).head(10)

print("\n=== CONTOH HASIL PREDIKSI PROBABILITAS ===")
print(df_hasil)


=== CONTOH HASIL PREDIKSI PROBABILITAS ===
      Aktual  Prediksi  Probabilitas_Churn
437        0         0            0.000000
2280       0         1            0.786667
2235       0         0            0.090000
4460       0         0            0.280000
3761       0         0            0.000000
5748       0         0            0.416667
3568       0         0            0.393333
2976       0         0            0.110000
5928       0         0            0.006667
1639       1         0            0.460000


Kesimpulan :
Model punya kemampuan diskriminasi keseluruhan yang layak (AUC 0.82), tapi belum layak dipakai untuk keputusan retensi karena recall churn hanya 50%. Prioritas perbaikan: threshold tuning atau resampling (SMOTE), cross-validation, dan analisis feature importance sebelum model ini dianggap "berhasil."